# Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from robotblockset.tools import get_rbs_path, print_xml, print_xml_for_console

from robotblockset.transformations import rot_x, rot_y, rot_z, map_pose

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, actuators_for_joint

# Initialization

Define the folder where the MJCF models are

In [3]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"

Define objects to be added to scene

In [4]:
ROBOT = "panda"
ROBOT_NAME1 = "panda"
ROBOT_NAME2 = "panda1"

# Basic scene

Load the basic scene into which other models will be included.

In [5]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene.xml")
scene.copy_during_attach = True

Add copy of Target body

In [6]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 0, 0], axisangle=[0, 0, 1, 0])
attachment_frame.attach_body(scene.body("Target"), "", "1")
scene.site("Target").rgba = [1, 0, 1, 1]
scene.site("Target1").rgba = [0, 1, 1, 1]
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-Target1


In [7]:
# print_xml(scene.to_xml())

# Robots

## Select first robot

Load robot specification

In [8]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)


Body Tree for "world"
-base
--link0
---link1 (Joints: joint1-HINGE[Actuator: pos_joint1])
----link2 (Joints: joint2-HINGE[Actuator: pos_joint2])
-----link3 (Joints: joint3-HINGE[Actuator: pos_joint3])
------link4 (Joints: joint4-HINGE[Actuator: pos_joint4])
-------link5 (Joints: joint5-HINGE[Actuator: pos_joint5])
--------link6 (Joints: joint6-HINGE[Actuator: pos_joint6])
---------link7 (Joints: joint7-HINGE[Actuator: pos_joint7])
----------flange
-----------panda_hand
------------left_finger (Joints: finger_joint1-SLIDE)
------------right_finger (Joints: finger_joint2-SLIDE)


Make robot movable using mocap feature.

In [9]:
robot_spec.body("base").mocap = True

Prepare data for model keyframes

In [10]:
robot1_ctrl = np.array(robot_spec.keys[0].ctrl)
robot1_qpos = np.array(robot_spec.keys[0].qpos)

### Attach robot to ground

Define the frame in the world where the robot will be attached.

In [11]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 0, 0], axisangle=[0, 0, 1, 0 * np.pi / 2])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT_NAME1}_")

Rename base bode of the robot body tree (needed for automatic hasndle generation)

In [12]:
scene.body(f"{ROBOT_NAME1}_base").name = ROBOT_NAME1

Optinally exclude contacts between robot base and table

In [13]:
# scene.add_exclude(bodyname1=f"{ROBOT_NAME1}", bodyname2="ground")

## Select second robot

The second robot is the same as the first

Prepare data for model keyframes

In [14]:
robot2_ctrl = np.array(robot_spec.keys[0].ctrl)
robot2_qpos = np.array(robot_spec.keys[0].qpos)

### Attach robot to ground

Define the frame in the world where the robot will be attached.

In [15]:
attachment_frame = scene.worldbody.add_frame(pos=[0, 1, 0], axisangle=[0, 0, 1, 0])
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT_NAME2}_")

Rename base bode of the robot body tree (needed for automatic hasndle generation)

In [16]:
scene.body(f"{ROBOT_NAME2}_base").name = ROBOT_NAME2

Optinally exclude contacts between robot base and table

In [17]:
# scene.add_exclude(bodyname1=f"{ROBOT_NAME2}", bodyname2="ground")

Check the body tree

In [18]:
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-Target1
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_panda_hand
------------panda_left_finger (Joints: panda_finger_joint1-SLIDE)
------------panda_right_finger (Joints: panda_finger_joint2-SLIDE)
-panda1
--panda1_link0
---panda1_link1 (Joints: panda1_joint1-HINGE[Actuator: panda1_pos_joint1])
----panda1_link2 (Joints: panda1_joint2-HINGE[Actuator: panda1_pos_joint2])
-----panda1_link3 (Joints: panda1_joint3-HINGE[Actuator: 

### Prepare MJCF XML file

Redefine MJCF keys. Delete all keys and define Key 0 as the initial configuration

In [19]:
scene.keys

[]

In [20]:
_tmp = scene.to_xml()
while len(scene.keys)>1:
    scene.delete(scene.keys[-1])
for k in scene.keys:
    k.ctrl = np.concatenate([robot1_ctrl, robot2_ctrl, np.zeros(len(scene.actuators) - len(robot1_ctrl) - len(robot2_ctrl))])

Save MJCF scene to XML

In [21]:
scene.modelname = f"{ROBOT}_2x_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{scene.modelname}.xml", "w") as f:
    f.write(scene.to_xml())